In [ ]:
using LowLevelFEM

In [ ]:
structured_rect_mesh()

mat = Material("surface")
Pu = Problem([mat], type=:VectorField, dim=2, field=:u, rhs_field=:f)
PT = Problem([mat], type=:ScalarField, dim=2, field=:T, rhs_field=:q)

In [ ]:
E = mat.E
ν = mat.ν
k = mat.k
α = mat.α
κ = mat.κ

D = E / (1 + ν^2) * [1 ν 0; ν 1 0; 0 0 (1-ν)/2]

In [ ]:
Km = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu))
Km[:, :]

In [ ]:
fm = ∫(Pu ⋅ [100, 0], Γ="right")

In [ ]:
Kh = ∫(Grad(PT) ⋅ k ⋅ Grad(PT))
Kh[:, :]

In [ ]:
q = ∫(PT ⋅ 500, Γ="right")

In [ ]:
αI = [1; 1; 0;;] * α

In [ ]:
KuT = ∫(SymGrad(Pu) ⋅ D ⋅ αI ⋅ PT)
KuT[:, :]

In [ ]:
T0 = 0
fuT = ∫(SymGrad(Pu) ⋅ D ⋅ αI ⋅ T0)

In [ ]:
supp1 = BoundaryCondition("left", problem=Pu, ux=0)
supp2 = BoundaryCondition("bottom", problem=Pu, uy=0)

suppT = BoundaryCondition("left", problem=PT, T=0)

In [ ]:
KuT.model

In [ ]:
K = SystemMatrix([Km -KuT'; 0*KuT Kh])
K[:, :]

In [ ]:
F = SystemVector([fm + fuT, q])

In [ ]:
u, T = solveField(K, F, support=[supp1, supp2, suppT])

In [ ]:
showDoFResults(u, name="u", factor=1, visible=true)

In [ ]:
showDoFResults(T, name="T")

In [ ]:
εx = ∂x(u[1])
εy = ∂y(u[2])
γxy = ∂y(u[1]) + ∂x(u[2])

ε = [εx, εy, γxy]

In [ ]:
T = nodesToElements(T)
εth = [α * T, α * T, 0]

In [ ]:
σ = D * (ε - εth)

In [ ]:
σx = σ[1]
σy = σ[2]
τxy = σ[3]

In [ ]:
showElementResults(σx, name="σx")
showElementResults(σy, name="σy")
showElementResults(τxy, name="τxy")

In [ ]:
openPostProcessor()